In [ ]:
import io, os, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import freqz, lfilter, butter, filtfilt, sosfiltfilt
import torch

warnings.filterwarnings("ignore", message=".*NumPy.*")

# ── Project imports (dataset utilities + model arch) ──────────────────────────
PROJECT_ROOT = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/MeMo-IK-ID")
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from dataset import IK_DOF_NAMES, MOMENT_NAMES, _compute_velocity
    _dataset_ok = True
    print("✓ dataset utilities loaded")
except ImportError as _e:
    print(f"[WARN] dataset import failed: {_e}  — falling back to np.gradient for velocity")
    _dataset_ok = False
    IK_DOF_NAMES = [
        "pelvis_tilt", "pelvis_list", "pelvis_rotation",
        "pelvis_tx", "pelvis_ty", "pelvis_tz",
        "hip_flexion_r", "hip_adduction_r", "hip_rotation_r",
        "knee_angle_r", "ankle_angle_r", "subtalar_angle_r", "mtp_angle_r",
        "hip_flexion_l", "hip_adduction_l", "hip_rotation_l",
        "knee_angle_l", "ankle_angle_l", "subtalar_angle_l", "mtp_angle_l",
        "lumbar_extension", "lumbar_bending", "lumbar_rotation",
    ]
    def _compute_velocity(pos: np.ndarray, time: np.ndarray) -> np.ndarray:
        return np.gradient(pos, time, axis=0)

try:
    from model import TCN
    print("✓ TCN model loaded")
except ImportError as _e:
    print(f"[ERROR] model import failed: {_e}")
    raise

## 1 · Configuration

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
OPENSIM_ROOT = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/opensim-processing/data/AB03_Ilseung")
NPZ_ROOT     = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/MeMo-IK-ID")

# ── model checkpoint (the main thing to swap when comparing models) ───────────
CHECKPOINT = PROJECT_ROOT / "runs/0509_ik_id_all/best_model.pt"

# ── subject parameters ─────────────────────────────────────────────────────────
SUBJECT_MASS_KG = 84.4      # un-normalise model output (N·m/kg) → N·m

# ── mocap analog sample rate (Hz) ─────────────────────────────────────────────
MOCAP_FS = 1000.0

# ── Output LPF — must match controller YAML (out_lpf_hz / out_lpf_order) ──────
# Applied CAUSALLY to the offline predictions to match what the controller emits.
MODEL_OUT_LPF_CUTOFF_HZ = 6.0
MODEL_OUT_LPF_ORDER     = 4

# ── ID zero-phase filter — matched bandwidth for a fair comparison ─────────────
ID_LPF_CUTOFF_HZ = 6.0
ID_LPF_ORDER     = 4

# ── per-exo timing shift (t_id_eval = t_npz_aligned − FIFO − extra − out-LPF τ_g) ──
# See compare_opensim_id_vs_model.ipynb for derivation.
MODEL_ID_EXTRA_SHIFT_S = {
    "hip-exo":  0.000,   # FIFO (0.12 s) covers model lag
    "knee-exo": -0.24,   # tune if still misaligned
}

# ── exo torque subtraction — which ID channels to correct per exo type ─────────
# List of (id_channel_index, npz_key, sign).
#   id_channel_index : index into OUTPUT_LABELS / ID_COLS (0=hip_r, 1=knee_r, 2=ankle_r,
#                      3=hip_l, 4=knee_l, 5=ankle_l)
#   npz_key          : key in the NPZ log for the actual motor command [N·m]
#   sign             : +1 if npz_key and OpenSim moment share the same sign convention
#                      (subtract directly); -1 if one is negated relative to the other.
EXO_TORQUE_CHANNELS = {
    "hip-exo":  [
        (0, "applied_R", +1),   # hip_flexion_r_moment − applied_R
        (3, "applied_L", +1),   # hip_flexion_l_moment − applied_L
    ],
    "knee-exo": [
        (1, "cmd_R",     +1),   # knee_angle_r_moment − cmd_R  (verify sign)
        (4, "cmd_L",     +1),   # knee_angle_l_moment − cmd_L  (verify sign)
    ],
}

# ── model output channels (must match checkpoint training order) ───────────────
OUTPUT_LABELS = [
    ("hip_flexion_r",  "Hip Flexion R"),
    ("knee_angle_r",   "Knee Angle R"),
    ("ankle_angle_r",  "Ankle Angle R"),
    ("hip_flexion_l",  "Hip Flexion L"),
    ("knee_angle_l",   "Knee Angle L"),
    ("ankle_angle_l",  "Ankle Angle L"),
]
N_OUT = len(OUTPUT_LABELS)

# Corresponding OpenSim ID .sto column names (N·m; same order as OUTPUT_LABELS)
ID_COLS = [
    "hip_flexion_r_moment",
    "knee_angle_r_moment",
    "ankle_angle_r_moment",
    "hip_flexion_l_moment",
    "knee_angle_l_moment",
    "ankle_angle_l_moment",
]

# ── colour palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "id":       "#2196F3",
    "model":    "#F44336",
    "residual": "#9C27B0",
    "gpio_m":   "#90A4AE",
    "gpio_n":   "#FF9800",
}
CHAN_COLORS_MODEL = ["#E53935", "#D81B60", "#8E24AA", "#1E88E5", "#039BE5", "#00ACC1"]
CHAN_COLORS_ID    = ["#EF9A9A", "#F48FB1", "#CE93D8", "#90CAF9", "#80DEEA", "#80CBC4"]

# ── trial discovery (same regex as compare_opensim_id_vs_model.ipynb) ─────────
TRIAL_STEM_RE = re.compile(
    r"^.+_(?P<joint>hip|knee)_(?P<speed>\d+p\d+mps)_(?P<code>lg|ra|rd)_exo_on$",
    re.IGNORECASE,
)


def trial_params_from_stem(stem: str):
    """Map NPZ basename → (exo_folder, opensim_cond_stem, id_moment_col, id_sign, delay_s) or None."""
    m = TRIAL_STEM_RE.match(stem)
    if not m:
        return None
    joint = m.group("joint").lower()
    speed = m.group("speed")
    code  = m.group("code").upper()
    cond  = f"{code}_{speed}"
    if joint == "hip":
        return ("hip-exo", cond, "hip_flexion_r_moment", +1, 0.12)
    return ("knee-exo", cond, "knee_angle_r_moment", -1, 0.00)


def discover_trials(npz_root: Path) -> dict:
    out = {}
    for p in sorted(npz_root.glob("*.npz")):
        meta = trial_params_from_stem(p.stem)
        if meta is not None:
            out[p.stem] = meta
    return out


TRIALS = discover_trials(NPZ_ROOT)
subject_name = Path(OPENSIM_ROOT).stem.split("_")[0].lower()
TRIALS = {s: m for s, m in TRIALS.items() if subject_name in s.lower()}

print(f"Trial stem regex: {TRIAL_STEM_RE.pattern}")
print(f"Checkpoint: {'✓' if CHECKPOINT.exists() else '✗'}  {CHECKPOINT}")
print("\nTrial manifest:")
for stem, (exo, cond, col, sign, delay) in TRIALS.items():
    npz_ok   = (NPZ_ROOT / f"{stem}.npz").exists()
    ik_ok    = (OPENSIM_ROOT / exo / "ik"    / f"{cond}_ik.mot").exists()
    id_ok    = (OPENSIM_ROOT / exo / "id"    / f"{cond}_id.sto").exists()
    mocap_ok = (OPENSIM_ROOT / exo / "mocap" / f"{cond}.csv").exists()
    status = "✓" if all([npz_ok, ik_ok, id_ok, mocap_ok]) else "✗"
    print(f"  {status}  npz={'✓' if npz_ok else '✗'}  ik={'✓' if ik_ok else '✗'}  "
          f"id={'✓' if id_ok else '✗'}  mocap={'✓' if mocap_ok else '✗'}  "
          f"delay={delay:.2f}s  {stem}")
if not TRIALS:
    print("  (no trials found — check NPZ_ROOT or TRIAL_STEM_RE)")

## 2 · Helper functions

In [ ]:
def parse_opensim_file(path: Path) -> pd.DataFrame:
    """Read an OpenSim .sto or .mot file; returns a DataFrame indexed by time."""
    with open(path) as fh:
        for i, line in enumerate(fh):
            if line.strip() == "endheader":
                header_end = i
                break
    df = pd.read_csv(path, sep=r"\s+", skiprows=header_end + 1)
    return df.set_index("time")


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS) -> tuple:
    """Parse a Vicon analog CSV (5-row header); return (time_s, gpio_voltage)."""
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0,
                     low_memory=False, on_bad_lines="skip")
    df = df[pd.to_numeric(df["Frame"], errors="coerce").notna()].copy()
    df["jet"] = pd.to_numeric(df["jet"], errors="coerce")
    df = df.dropna(subset=["jet"]).reset_index(drop=True)
    return np.arange(len(df)) / fs, df["jet"].to_numpy(dtype=float)


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> int | None:
    """Index of the first sample immediately after a falling edge."""
    above = signal > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


# ── causal cascaded-EMA LPF (replicates controller _CausalLowPass) ────────────
def _cascaded_ema_ba(fs_hz: float, cutoff_hz: float, order: int):
    dt    = 1.0 / float(fs_hz)
    tau   = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)
    b = np.array([1.0])
    a = np.array([1.0])
    for _ in range(max(1, int(order))):
        b = np.convolve(b, [alpha])
        a = np.convolve(a, [1.0, -(1.0 - alpha)])
    return b, a


def causal_lpf(signal: np.ndarray, fs: float,
               cutoff_hz: float = MODEL_OUT_LPF_CUTOFF_HZ,
               order: int = MODEL_OUT_LPF_ORDER) -> np.ndarray:
    """Apply cascaded-EMA causal LPF to a 1-D signal (matches controller)."""
    dt    = 1.0 / float(fs)
    tau   = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)
    b = np.array([alpha])
    a = np.array([1.0, -(1.0 - alpha)])
    y = signal.astype(float).copy()
    for _ in range(order):
        zi = np.array([y[0] * (1.0 - alpha)])
        y, _ = lfilter(b, a, y, zi=zi)
    return y


def causal_lpf_multichannel(signals: np.ndarray, fs: float,
                             cutoff_hz: float = MODEL_OUT_LPF_CUTOFF_HZ,
                             order: int = MODEL_OUT_LPF_ORDER) -> np.ndarray:
    """Apply causal_lpf independently to each column of an (T, C) array."""
    return np.stack([causal_lpf(signals[:, c], fs, cutoff_hz, order)
                     for c in range(signals.shape[1])], axis=1)


def model_output_lpf_group_delay_s(fs_hz: float,
                                    cutoff_hz: float = MODEL_OUT_LPF_CUTOFF_HZ,
                                    order: int = MODEL_OUT_LPF_ORDER) -> float:
    """Median group delay (s) of the causal output LPF via freqz + phase derivative."""
    b, a = _cascaded_ema_ba(fs_hz, cutoff_hz, order)
    try:
        w_hz, h = freqz(b, a, worN=8192, fs=float(fs_hz))
    except TypeError:
        try:
            w_hz, h = freqz(b, a, 8192, fs=float(fs_hz))
        except TypeError:
            w_rad, h = freqz(b, a, 8192)
            w_hz = w_rad * float(fs_hz) / (2.0 * np.pi)
    phi  = np.unwrap(np.angle(h))
    dphi = np.gradient(phi) / (np.gradient(w_hz) + 1e-15)
    gd   = -dphi / (2.0 * np.pi)
    ok   = np.isfinite(gd) & (w_hz >= 0.2) & (w_hz <= min(8.0, 0.45 * float(fs_hz)))
    return float(np.nanmedian(gd[ok])) if np.any(ok) else 0.0


# ── zero-phase Butterworth LPF for GT ID ──────────────────────────────────────
def zpf_id_1d(signal: np.ndarray, fs: float,
               cutoff_hz: float = ID_LPF_CUTOFF_HZ,
               order: int = ID_LPF_ORDER) -> np.ndarray:
    b, a = butter(order, cutoff_hz / (fs / 2.0), btype="low")
    return filtfilt(b, a, signal.astype(float))


def zpf_id_multichannel(signals: np.ndarray, fs: float) -> np.ndarray:
    return np.stack([zpf_id_1d(signals[:, c], fs)
                     for c in range(signals.shape[1])], axis=1)


# ── zero-phase Butterworth LPF for IK preprocessing (matches dataset.py) ──────
def lowpass_zero_phase(data: np.ndarray, time: np.ndarray,
                        cutoff_hz: float = None, order: int = None) -> np.ndarray:
    _cut = cutoff_hz if cutoff_hz is not None else ID_LPF_CUTOFF_HZ
    _ord = order     if order     is not None else ID_LPF_ORDER
    fs      = 1.0 / float(np.median(np.diff(time.astype(np.float64))))
    nyquist = 0.5 * fs
    if _cut <= 0 or _cut >= nyquist or data.shape[0] < 8:
        return data
    sos = butter(_ord, _cut / nyquist, btype="low", output="sos")
    return sosfiltfilt(sos, data.astype(np.float64), axis=0)


print("Helpers defined.")

## 3 · Load checkpoint

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
cfg  = ckpt["model_config"]

model = TCN(
    n_input_channels=cfg["n_input_channels"],
    n_output_channels=cfg["n_output_channels"],
    hidden_channels=cfg["hidden_channels"],
    n_blocks=cfg["n_blocks"],
    kernel_size=cfg["kernel_size"],
    dropout=cfg["dropout"],
).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

WINDOW_SIZE   = int(ckpt["window_size"])
INPUT_INDICES = list(ckpt["input_indices"])    # e.g. [6,9,10,13,16,17]
ROLLOUT_STEP  = 1

# IK preprocessing LPF params (fall back to ID_LPF values if config.json absent)
_cfg_json = CHECKPOINT.parent / "config.json"
if _cfg_json.exists():
    import json as _json
    _jcfg = _json.load(open(_cfg_json))
    IK_LPF_CUTOFF_HZ = float(_jcfg.get("lowpass_cutoff_hz", ID_LPF_CUTOFF_HZ))
    IK_LPF_ORDER     = int(_jcfg.get("lowpass_order",       ID_LPF_ORDER))
else:
    IK_LPF_CUTOFF_HZ = ID_LPF_CUTOFF_HZ
    IK_LPF_ORDER     = ID_LPF_ORDER

H = len(INPUT_INDICES) // 2
INPUT_IDX_R = INPUT_INDICES[:H]   # right-side DOF indices into IK_DOF_NAMES
INPUT_IDX_L = INPUT_INDICES[H:]   # left-side DOF indices

norm    = ckpt["normalization"]
POS_MEAN = np.array(norm["pos_mean"])
POS_STD  = np.array(norm["pos_std"])
VEL_MEAN = np.array(norm["vel_mean"])
VEL_STD  = np.array(norm["vel_std"])

print(f"Checkpoint : {CHECKPOINT}")
print(f"Architecture: {cfg['n_input_channels']}→{cfg['n_output_channels']} ch, "
      f"window={WINDOW_SIZE}, blocks={cfg['n_blocks']}, hidden={cfg['hidden_channels']}")
print(f"Device     : {DEVICE}")
print(f"Input DOFs R : {[IK_DOF_NAMES[i] for i in INPUT_IDX_R]}")
print(f"Input DOFs L : {[IK_DOF_NAMES[i] for i in INPUT_IDX_L]}")
print(f"IK LPF     : {IK_LPF_CUTOFF_HZ} Hz, order {IK_LPF_ORDER} (zero-phase)")

## 4 · Inference functions

In [ ]:
@torch.no_grad()
def causal_infer_unilateral(pos_in: np.ndarray, vel_in: np.ndarray) -> np.ndarray:
    """
    Causal sliding-window TCN inference for one side (R or L).
    pos_in, vel_in : (T, H) arrays — H = len(INPUT_IDX_R) = len(INPUT_IDX_L).
    Returns (T, n_out_unilateral) in N·m/kg.
    """
    x = np.concatenate([pos_in, vel_in], axis=1).astype(np.float32)  # (T, 2H)
    n     = x.shape[0]
    W     = WINDOW_SIZE
    n_out = cfg["n_output_channels"]
    pred  = np.zeros((n, n_out), dtype=np.float64)

    def _fwd(start: int) -> np.ndarray:
        arr = np.ascontiguousarray(x[start:start + W].T)   # (2H, W)
        xw  = torch.from_numpy(arr).unsqueeze(0).to(DEVICE)  # (1, 2H, W)
        return model(xw).squeeze(0).detach().cpu().numpy().T  # (W, n_out)

    pw0 = _fwd(0)
    for g in range(min(W, n)):
        pred[g] = pw0[g]
    for start in range(1, n - W + 1):
        pred[start + W - 1] = _fwd(start)[W - 1]

    return pred.astype(np.float32)   # (T, n_out)


def run_inference(pos_all: np.ndarray, vel_all: np.ndarray) -> np.ndarray:
    """
    Bilateral inference via unilateral-paired strategy.
    pos_all, vel_all : (T, 23) — full IK position / velocity (radians, rad/s).
    Returns (T, 6) in N·m/kg: [hip_r, knee_r, ankle_r, hip_l, knee_l, ankle_l].
    """
    pred_r = causal_infer_unilateral(pos_all[:, INPUT_IDX_R], vel_all[:, INPUT_IDX_R])
    pred_l = causal_infer_unilateral(pos_all[:, INPUT_IDX_L], vel_all[:, INPUT_IDX_L])
    return np.concatenate([pred_r, pred_l], axis=1)   # (T, 6)


print("Inference functions defined.")

## 5 · Pre-load & sync all trials

For each trial this cell:
1. **GPIO sync** (NPZ ↔ mocap CSV falling edge) to get `t_offset` mapping NPZ time → mocap/ID frame.
2. **Delay correction**: `t_id_eval = t_npz_aligned − FIFO − extra − out-LPF τ_g`  (same formula as `compare_opensim_id_vs_model.ipynb`).
3. **IK preprocessing**: load `.mot`, deg→rad, zero-phase LPF, velocity, zero-phase LPF again (matches `dataset.py` pipeline).
4. **Resample IK → NPZ grid**: interpolate (pos, vel) from IK time axis onto `t_npz_aligned` so the model sees inputs at the controller's real-time sampling instants.
5. **Offline causal inference** on the resampled IK sequence → `pred_nmpkg (T_npz, 6)`.
6. **Apply causal output LPF** to predictions (matches controller `out_lpf_*` post-processing).
7. **GT ID**: load `.sto`, apply zero-phase 6 Hz Butterworth (same bandwidth as model), evaluate at `t_id_eval`.
8. **Subtract actual applied torque**: `applied_R`/`applied_L` (hip) or `cmd_R`/`cmd_L` (knee) interpolated from NPZ to `t_id_eval` — same actual motor commands the body experienced.
9. Cache everything in `TRIAL_DATA`.

In [ ]:
TRIAL_DATA = {}

for stem, (exo, cond, moment_col, id_sign, delay_s) in TRIALS.items():
    npz_path   = NPZ_ROOT     / f"{stem}.npz"
    ik_path    = OPENSIM_ROOT / exo / "ik"    / f"{cond}_ik.mot"
    id_path    = OPENSIM_ROOT / exo / "id"    / f"{cond}_id.sto"
    mocap_path = OPENSIM_ROOT / exo / "mocap" / f"{cond}.csv"

    if not all(p.exists() for p in (npz_path, ik_path, id_path, mocap_path)):
        missing = [str(p) for p in (npz_path, ik_path, id_path, mocap_path) if not p.exists()]
        print(f"[SKIP] {stem} — missing: {missing}")
        continue

    print(f"Loading {stem} …", end=" ", flush=True)

    # ── 1. NPZ ────────────────────────────────────────────────────────────────
    npz      = np.load(npz_path)
    t_npz    = npz["time"]
    gpio_npz = npz["GPIO"]
    fs_npz   = 1.0 / float(np.median(np.diff(t_npz.astype(float))))

    # ── 2. GPIO sync ──────────────────────────────────────────────────────────
    t_mocap, gpio_mocap = parse_mocap_csv(mocap_path)
    g_range = gpio_mocap.max() - gpio_mocap.min()
    gpio_mocap_norm = (gpio_mocap - gpio_mocap.min()) / g_range if g_range > 0 else gpio_mocap

    idx_npz   = first_falling_edge(gpio_npz)
    idx_mocap = first_falling_edge(gpio_mocap_norm)
    if idx_npz is None or idx_mocap is None:
        print("[WARN] no GPIO falling edge — skipping")
        continue
    t_offset      = t_mocap[idx_mocap] - t_npz[idx_npz]
    t_npz_aligned = t_npz + t_offset   # NPZ time in mocap/ID frame

    # ── 3. Delay correction ───────────────────────────────────────────────────
    tau_out       = model_output_lpf_group_delay_s(fs_npz)
    extra_shift_s = MODEL_ID_EXTRA_SHIFT_S.get(exo, 0.0)
    t_id_eval     = t_npz_aligned - delay_s - extra_shift_s - tau_out

    # ── 4. GT ID: load, zero-phase LPF, all 6 channels ───────────────────────
    id_df  = parse_opensim_file(id_path)
    t_id   = id_df.index.to_numpy(float)
    id_fs  = 1.0 / float(np.median(np.diff(t_id)))
    id_raw = np.zeros((len(t_id), N_OUT), dtype=np.float64)
    for c, col in enumerate(ID_COLS):
        if col in id_df.columns:
            id_raw[:, c] = id_df[col].to_numpy(float)
        else:
            id_raw[:, c] = np.nan
    id_filt = zpf_id_multichannel(id_raw, fs=id_fs)   # (T_id, 6)

    # ── 5. Overlap mask ───────────────────────────────────────────────────────
    mask     = (t_id_eval >= t_id[0]) & (t_id_eval <= t_id[-1])
    t_common = t_npz_aligned[mask]

    # ── 6. IK: load, degrees→radians, zero-phase LPF, velocity, LPF again ────
    ik_df      = parse_opensim_file(ik_path)
    t_ik       = ik_df.index.to_numpy(float)
    ik_fs_nat  = 1.0 / float(np.median(np.diff(t_ik)))
    eff_step   = ROLLOUT_STEP if ik_fs_nat > 150.0 else 1
    if eff_step > 1:
        t_ik = t_ik[::eff_step]

    pos_deg = np.zeros((len(t_ik), len(IK_DOF_NAMES)), dtype=np.float64)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            raw_col = ik_df[name].to_numpy(float)
            if eff_step > 1:
                raw_col = raw_col[::eff_step]
            pos_deg[:, j] = raw_col

    pos_rad  = np.deg2rad(pos_deg)
    pos_filt = lowpass_zero_phase(pos_rad, t_ik,
                                   cutoff_hz=IK_LPF_CUTOFF_HZ, order=IK_LPF_ORDER)
    vel_raw  = _compute_velocity(pos_filt, t_ik)
    vel_filt = lowpass_zero_phase(vel_raw, t_ik,
                                   cutoff_hz=IK_LPF_CUTOFF_HZ, order=IK_LPF_ORDER)

    # ── 7. Resample IK → t_npz_aligned (full sequence for causal context) ─────
    _ik_interp_pos = interp1d(t_ik, pos_filt, axis=0, kind="linear",
                               bounds_error=False, fill_value="extrapolate")
    _ik_interp_vel = interp1d(t_ik, vel_filt, axis=0, kind="linear",
                               bounds_error=False, fill_value="extrapolate")
    pos_npz = _ik_interp_pos(t_npz_aligned)   # (T_npz, 23)
    vel_npz = _ik_interp_vel(t_npz_aligned)   # (T_npz, 23)

    # ── 8. Offline causal inference on the full NPZ-rate sequence ─────────────
    pred_nmpkg = run_inference(pos_npz, vel_npz)        # (T_npz, 6)  N·m/kg
    pred_nm    = pred_nmpkg * SUBJECT_MASS_KG           # (T_npz, 6)  N·m
    pred_nm    = causal_lpf_multichannel(pred_nm, fs=fs_npz)  # causal out-LPF

    # ── 9. Extract predictions at the overlap ─────────────────────────────────
    pred_common = pred_nm[mask]   # (T_overlap, 6)

    # ── 10. GT ID at t_id_eval (delay-corrected) ──────────────────────────────
    id_eval = np.stack([
        interp1d(t_id, id_filt[:, c], kind="linear",
                 bounds_error=False, fill_value=np.nan)(t_id_eval[mask])
        for c in range(N_OUT)
    ], axis=1)   # (T_overlap, 6)

    # ── 11. Subtract actual applied exo torque from the relevant ID channels ──
    #   Build a (T_npz, 6) array of actual motor commands in Nm, then interpolate
    #   to t_id_eval so the subtraction is co-temporal with the ID evaluation.
    exo_tau_npz = np.zeros((len(t_npz_aligned), N_OUT), dtype=np.float64)
    for id_ch, npz_key, sign in EXO_TORQUE_CHANNELS.get(exo, []):
        if npz_key in npz.files:
            exo_tau_npz[:, id_ch] = sign * np.asarray(npz[npz_key], dtype=float)
    exo_torque = interp1d(
        t_npz_aligned, exo_tau_npz, axis=0,
        kind="linear", bounds_error=False, fill_value=0.0,
    )(t_id_eval[mask])   # (T_overlap, 6)

    id_net = id_eval - exo_torque   # (T_overlap, 6)  — biological moment

    # ── 12. Per-channel metrics ───────────────────────────────────────────────
    metrics = []
    for c in range(N_OUT):
        v = ~np.isnan(id_net[:, c])
        if v.sum() > 1:
            rmse = float(np.sqrt(np.mean((pred_common[v, c] - id_net[v, c]) ** 2)))
            r2   = float(np.corrcoef(pred_common[v, c], id_net[v, c])[0, 1]) ** 2
        else:
            rmse, r2 = np.nan, np.nan
        metrics.append({"rmse": rmse, "r2": r2})

    # ── 13. Store ─────────────────────────────────────────────────────────────
    # Determine which NPZ keys were actually subtracted
    subtracted_keys = [
        (id_ch, npz_key, sign)
        for (id_ch, npz_key, sign) in EXO_TORQUE_CHANNELS.get(exo, [])
        if npz_key in npz.files
    ]
    TRIAL_DATA[stem] = {
        "stem":            stem,
        "exo":             exo,
        "cond":            cond,
        "delay_s":         delay_s,
        "extra_shift_s":   extra_shift_s,
        "tau_out_s":       tau_out,
        "fs_npz":          fs_npz,
        "t_offset":        t_offset,
        "t":               t_common,
        "t_rel":           t_common - t_common[0],
        "pred_nm":         pred_common,       # (T, 6)
        "id_nm_gross":     id_eval,           # (T, 6) before τ_exo subtraction
        "exo_torque_nm":   exo_torque,        # (T, 6)
        "id_nm":           id_net,            # (T, 6) = GT ID − applied τ_exo
        "subtracted_keys": subtracted_keys,
        "metrics":         metrics,
        "t_mocap_rel":     t_mocap - t_mocap[0],
        "gpio_mocap_norm": gpio_mocap_norm,
        "t_npz_rel":       t_npz - t_npz[0],
        "gpio_npz":        gpio_npz,
    }

    dur       = float(t_common[-1] - t_common[0])
    mean_r2   = float(np.nanmean([m["r2"]   for m in metrics]))
    mean_rmse = float(np.nanmean([m["rmse"] for m in metrics]))
    total_ms  = (delay_s + extra_shift_s + tau_out) * 1000
    keys_str  = ", ".join(f"ch{ic}←NPZ['{k}']×{sg:+d}" for ic, k, sg in subtracted_keys)
    print(
        f"ok  ({dur:.1f} s, t_off={t_offset:+.3f} s, "
        f"Δt={delay_s*1000:.0f}+{extra_shift_s*1000:.0f}+{tau_out*1000:.1f}={total_ms:.0f} ms; "
        f"mean R²={mean_r2:.3f}, RMSE={mean_rmse:.2f} N·m; τ_exo: {keys_str or 'none'})"
    )

print(f"\nLoaded {len(TRIAL_DATA)} / {len(TRIALS)} trials.")
print(f"Checkpoint: {CHECKPOINT.name}")

## 6 · Interactive comparison

3 × 2 grid (rows = hip / knee / ankle; columns = R / L). Each panel shows **offline model** (GT IK → checkpoint → N·m) vs **GT ID − applied τ_exo** (GPIO-synced OpenSim ID minus actual motor commands from NPZ, zero-phase filtered).

In [ ]:
_first_stem = next(iter(TRIAL_DATA))
_first_d    = TRIAL_DATA[_first_stem]

trial_dd = widgets.Dropdown(
    options=list(TRIAL_DATA.keys()), value=_first_stem,
    description="Trial:",
    layout=widgets.Layout(width="640px"), style={"description_width": "80px"},
)
time_sl = widgets.FloatRangeSlider(
    value=[_first_d["t_rel"][0], _first_d["t_rel"][-1]],
    min=_first_d["t_rel"][0], max=_first_d["t_rel"][-1],
    step=0.5, description="Time (s):",
    layout=widgets.Layout(width="720px"), style={"description_width": "80px"},
    continuous_update=False, readout_format=".1f",
)
resid_btn = widgets.ToggleButton(
    value=False, description="Show residuals",
    icon="signal", layout=widgets.Layout(width="160px"),
)
gpio_btn = widgets.ToggleButton(
    value=False, description="Show GPIO",
    icon="signal", layout=widgets.Layout(width="140px"),
)
out = widgets.Output()


def _draw(stem: str, t0: float, t1: float,
          show_residuals: bool, show_gpio: bool) -> None:
    d    = TRIAL_DATA[stem]
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t    = d["t_rel"][mask]
    pred = d["pred_nm"][mask]    # (T_win, 6)
    id_m = d["id_nm"][mask]      # (T_win, 6)

    n_joints = 3
    n_sides  = 2
    n_moment_rows = n_joints * (2 if show_residuals else 1)
    n_rows   = n_moment_rows + (1 if show_gpio else 0)
    row_h    = [2.2] * n_moment_rows + ([1.6] if show_gpio else [])

    fig, axs = plt.subplots(
        n_moment_rows + (1 if show_gpio else 0), n_sides,
        figsize=(16, sum(row_h) + 0.6),
        gridspec_kw={"height_ratios": row_h, "hspace": 0.55, "wspace": 0.25},
        sharex=True,
    )
    if axs.ndim == 1:
        axs = axs.reshape(-1, n_sides)

    zpf_tag  = f"{ID_LPF_CUTOFF_HZ:.0f} Hz ZPF (Butterworth)"
    lpf_tag  = f"{MODEL_OUT_LPF_CUTOFF_HZ:.0f} Hz LPF (causal)"
    tau_out  = float(d.get("tau_out_s", 0.0))
    extra_ms = float(d.get("extra_shift_s", 0.0)) * 1000
    total_ms = d["delay_s"] * 1000 + extra_ms + tau_out * 1000

    has_exo  = bool(d.get("subtracted_keys"))
    id_label = (f"GT ID − τ_exo ({zpf_tag})" if has_exo
                else f"GT ID ({zpf_tag})")
    res_ttl  = "Model − (ID − τ_exo)" if has_exo else "Model − ID"

    joint_names = ["Hip Flexion", "Knee Angle", "Ankle Angle"]
    side_names  = ["Right", "Left"]

    for j in range(n_joints):
        for s in range(n_sides):
            c = j + s * n_joints   # 0-2 = R, 3-5 = L
            m = d["metrics"][c]

            ax_row = j * (2 if show_residuals else 1)
            ax = axs[ax_row, s]

            ax.plot(t, id_m[:, c], color=CHAN_COLORS_ID[c], lw=2.2, label=id_label)
            ax.plot(t, pred[:, c], color=CHAN_COLORS_MODEL[c], lw=1.8, ls="--",
                    label=f"Offline model × {SUBJECT_MASS_KG:.0f} kg ({lpf_tag})")
            ax.axhline(0, color="gray", lw=0.6, ls=":")
            ax.set_ylabel("N·m", fontsize=10)
            ax.set_title(
                f"{joint_names[j]} — {side_names[s]}\n"
                f"RMSE = {m['rmse']:.2f} N·m   R² = {m['r2']:.3f}",
                fontsize=9,
            )
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=9)
            if j == 0 and s == 0:
                ax.legend(fontsize=8, loc="upper right")

            if show_residuals:
                ax_r = axs[ax_row + 1, s]
                resid = pred[:, c] - id_m[:, c]
                ax_r.plot(t, resid, color=PALETTE["residual"], lw=1.2)
                ax_r.axhline(0, color="black", lw=0.7, ls=":")
                ax_r.set_ylabel("N·m", fontsize=9)
                ax_r.set_title(res_ttl, fontsize=8)
                ax_r.spines[["top", "right"]].set_visible(False)
                ax_r.tick_params(labelsize=9)

    for s in range(n_sides):
        axs[n_moment_rows - 1, s].set_xlabel("Time (s)", fontsize=10)

    if show_gpio:
        # Merge the two GPIO cells into a single row spanning both columns
        for s in range(n_sides):
            axs[-1, s].remove()
        ax_g = fig.add_subplot(n_rows, 1, n_rows)
        ax_g.plot(d["t_mocap_rel"], d["gpio_mocap_norm"],
                  color=PALETTE["gpio_m"], lw=1.0, alpha=0.8,
                  label="Mocap GPIO (normalised)")
        ax_g.plot(d["t_npz_rel"], d["gpio_npz"],
                  color=PALETTE["gpio_n"], lw=1.3, ls="--",
                  label="NPZ GPIO")
        ax_g.set_ylabel("Amplitude (a.u.)", fontsize=10)
        ax_g.set_xlabel("Time from recording start (s)", fontsize=10)
        ax_g.set_title(f"GPIO sync  (t_offset = {d['t_offset']:+.4f} s)", fontsize=9)
        ax_g.legend(fontsize=9, loc="upper right")
        ax_g.spines[["top", "right"]].set_visible(False)

    mean_r2   = float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)]))
    mean_rmse = float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)]))
    keys_str  = ", ".join(f"ch{ic}←'{k}'" for ic, k, _ in d.get("subtracted_keys", []))
    fig.suptitle(
        f"{stem}  |  Offline checkpoint vs GT ID {'− τ_exo' if has_exo else ''}\n"
        f"Δt = {d['delay_s']*1000:.0f} + {extra_ms:.0f} + {tau_out*1000:.1f} = {total_ms:.0f} ms  "
        f"| τ_exo keys: {keys_str or 'none'}  "
        f"| mean R² = {mean_r2:.3f}   mean RMSE = {mean_rmse:.2f} N·m",
        fontsize=10, fontweight="bold",
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


def _on_trial(change):
    d = TRIAL_DATA[change["new"]]
    time_sl.min   = float(d["t_rel"][0])
    time_sl.max   = float(d["t_rel"][-1])
    time_sl.value = [float(d["t_rel"][0]), float(d["t_rel"][-1])]
    _redraw()


def _redraw(*_):
    _draw(trial_dd.value, *time_sl.value, resid_btn.value, gpio_btn.value)


trial_dd.observe(_on_trial,   names="value")
time_sl.observe(_redraw,      names="value")
resid_btn.observe(_redraw,    names="value")
gpio_btn.observe(_redraw,     names="value")

controls = widgets.HBox(
    [trial_dd, resid_btn, gpio_btn],
    layout=widgets.Layout(align_items="center", gap="12px"),
)
display(controls, time_sl, out)
_redraw()

## 7 · Window statistics

Re-run after adjusting the slider.

In [ ]:
t0_s, t1_s = time_sl.value
stem_s      = trial_dd.value
d_s         = TRIAL_DATA[stem_s]
zpf_tag     = f"{ID_LPF_CUTOFF_HZ:.0f} Hz ZPF"

mask_s   = (d_s["t_rel"] >= t0_s) & (d_s["t_rel"] <= t1_s)
pred_s   = d_s["pred_nm"][mask_s]
id_s_win = d_s["id_nm"][mask_s]

has_exo  = bool(d_s.get("subtracted_keys"))
keys_str = ", ".join(f"ch{ic}←NPZ['{k}']" for ic, k, _ in d_s.get("subtracted_keys", []))

print(f"Trial      : {stem_s}")
print(f"Window     : {t0_s:.1f} – {t1_s:.1f} s  ({t1_s - t0_s:.1f} s)")
print(f"τ_exo keys : {keys_str or 'none'}")
print(f"Checkpoint : {CHECKPOINT.name}")
print()

_id_hdr = "peak ID−τ_exo" if has_exo else "peak ID"
print(f"{'Channel':<30} {'RMSE [N·m]':>10} {'R²':>7} {'peak model':>11} {_id_hdr:>14}")
print("-" * 78)
for c, (key, label) in enumerate(OUTPUT_LABELS):
    v = ~np.isnan(id_s_win[:, c])
    if v.sum() > 1:
        rmse_c = float(np.sqrt(np.mean((pred_s[v, c] - id_s_win[v, c]) ** 2)))
        r2_c   = float(np.corrcoef(pred_s[v, c], id_s_win[v, c])[0, 1]) ** 2
        pk_m   = float(np.nanmax(np.abs(pred_s[v, c])))
        pk_i   = float(np.nanmax(np.abs(id_s_win[v, c])))
    else:
        rmse_c = r2_c = pk_m = pk_i = np.nan
    print(f"  {label:<28} {rmse_c:>10.3f} {r2_c:>7.3f} {pk_m:>11.2f} {pk_i:>14.2f}")

valid_r2   = [c for c in range(N_OUT) if (~np.isnan(id_s_win[:, c])).sum() > 1]
mean_r2    = float(np.nanmean([
    float(np.corrcoef(pred_s[~np.isnan(id_s_win[:, c]), c],
                      id_s_win[~np.isnan(id_s_win[:, c]), c])[0, 1]) ** 2
    for c in valid_r2
])) if valid_r2 else np.nan
mean_rmse  = float(np.nanmean([
    float(np.sqrt(np.mean((pred_s[~np.isnan(id_s_win[:, c]), c]
                           - id_s_win[~np.isnan(id_s_win[:, c]), c]) ** 2)))
    for c in valid_r2
])) if valid_r2 else np.nan

print()
print(f"  {'Mean (all channels)':<28} {mean_rmse:>10.3f} {mean_r2:>7.3f}")

## 8 · All-trials summary

In [ ]:
rows = []
for stem, d in TRIAL_DATA.items():
    has_exo   = bool(d.get("subtracted_keys"))
    keys_str  = ", ".join(f"ch{ic}←'{k}'" for ic, k, _ in d.get("subtracted_keys", []))
    total_ms  = (d["delay_s"] + d["extra_shift_s"] + d["tau_out_s"]) * 1000
    row = {
        "trial":       stem,
        "exo":         d["exo"],
        "condition":   d["cond"],
        "checkpoint":  CHECKPOINT.name,
        "ID filter":   f"{ID_LPF_CUTOFF_HZ:.0f} Hz ZPF Butterworth",
        "GT ID":       "ID − τ_exo" if has_exo else "ID gross",
        "τ_exo keys":  keys_str or "none",
        "Δt total [ms]": round(total_ms, 1),
        "t_offset [s]": round(float(d["t_offset"]), 4),
        "overlap [s]":  round(float(d["t_rel"][-1] - d["t_rel"][0]), 1),
        "mean_RMSE [N·m]": round(float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)])), 2),
        "mean_R²":          round(float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)])), 3),
    }
    for c, (key, label) in enumerate(OUTPUT_LABELS):
        m = d["metrics"][c]
        row[f"RMSE_{key} [N·m]"] = round(m["rmse"], 2) if not np.isnan(m["rmse"]) else np.nan
        row[f"R²_{key}"]          = round(m["r2"],   3) if not np.isnan(m["r2"])   else np.nan
    rows.append(row)

summary_df = pd.DataFrame(rows)

# Print concise summary first, then display full table
print(f"Checkpoint: {CHECKPOINT}\n")
cols_compact = ["trial", "exo", "mean_RMSE [N·m]", "mean_R²", "Δt total [ms]", "GT ID", "τ_exo keys"]
print(summary_df[cols_compact].to_string(index=False))

summary_df